# Model Evaluation - CityFlow Traffic Tension Prediction

Comprehensive evaluation of Random Forest, XGBoost, and LightGBM models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.append('../src')
from train import preprocess_data

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

MODELS_DIR = '../models'
DATA_PATH = '../data/metro_traffic.csv'

## Load Data & Models

In [ ]:
df = pd.read_csv(DATA_PATH)

X, y, scaler_features, scaler_target = preprocess_data(df, fit=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_files = {
    'Random Forest': os.path.join(MODELS_DIR, 'random_forest_model.joblib'),
    'XGBoost':       os.path.join(MODELS_DIR, 'xgboost_model.joblib'),
    'LightGBM':      os.path.join(MODELS_DIR, 'lightgbm_model.joblib'),
}

missing = [name for name, path in model_files.items() if not os.path.exists(path)]
if missing:
    print(f"Missing models: {missing}")
    print("Run src/train.py first to generate models.")
else:
    models = {name: joblib.load(path) for name, path in model_files.items()}
    print(f"Loaded {len(models)} models")
    print(f"Test set: {X_test.shape[0]:,} samples | Features: {list(X_test.columns)}")

## Metrics Comparison

In [ ]:
results = {}
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R²': r2}

results_df = pd.DataFrame(results).T.sort_values('R²', ascending=False)
print(results_df.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R²']):
    colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
    bars = ax.bar(results_df.index, results_df[metric], color=colors, edgecolor='black', alpha=0.85)
    ax.set_title(f'{metric} by Model', fontweight='bold', fontsize=13)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax.tick_params(axis='x', rotation=10)

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Predictions vs Actual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, y_pred, alpha=0.3, s=8, color='steelblue')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Tension Score')
    ax.set_ylabel('Predicted Tension Score')
    ax.set_title(f'{name}\n(R² = {results[name]["R²"]:.4f})', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predictions vs Actual', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, (name, y_pred) in enumerate(predictions.items()):
    residuals = y_test.values - y_pred

    axes[0, col].scatter(y_pred, residuals, alpha=0.3, s=8, color='darkorange')
    axes[0, col].axhline(0, color='red', linestyle='--', linewidth=1.5)
    axes[0, col].set_xlabel('Predicted')
    axes[0, col].set_ylabel('Residual')
    axes[0, col].set_title(f'{name} - Residuals vs Predicted', fontweight='bold')
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].hist(residuals, bins=60, edgecolor='black', alpha=0.7, color='teal')
    axes[1, col].axvline(0, color='red', linestyle='--', linewidth=1.5)
    axes[1, col].set_xlabel('Residual')
    axes[1, col].set_ylabel('Frequency')
    axes[1, col].set_title(f'{name} - Error Distribution\n(mean={residuals.mean():.2f}, std={residuals.std():.2f})', fontweight='bold')
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Residual Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
feature_names = list(X_test.columns)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models.items()):
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_importances = importances[sorted_idx]

    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(sorted_features)))[::-1]
    bars = ax.barh(range(len(sorted_features)), sorted_importances, color=colors, edgecolor='black', alpha=0.85)
    ax.set_yticks(range(len(sorted_features)))
    ax.set_yticklabels(sorted_features)
    ax.set_xlabel('Importance')
    ax.set_title(f'{name}\nFeature Importance', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    for bar, val in zip(bars, sorted_importances):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

plt.suptitle('Feature Importance by Model', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Cross-Validation (k-Fold)

In [ ]:
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name}: R² = {scores.mean():.4f} ± {scores.std():.4f}  | folds: {np.round(scores, 4)}")

fig, ax = plt.subplots(figsize=(10, 5))

positions = range(len(cv_results))
for i, (name, scores) in enumerate(cv_results.items()):
    ax.boxplot(scores, positions=[i], widths=0.4, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.7),
               medianprops=dict(color='red', linewidth=2))
    ax.scatter([i]*len(scores), scores, color='black', zorder=5, s=30, alpha=0.7)

ax.set_xticks(positions)
ax.set_xticklabels(cv_results.keys())
ax.set_ylabel('R² Score')
ax.set_title('5-Fold Cross-Validation R² Scores', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Best Model Summary

In [ ]:
best_name = results_df.index[0]
best_model = models[best_name]
best_pred = predictions[best_name]

print("=" * 50)
print(f"  Best Model: {best_name}")
print("=" * 50)
print(f"  R²  : {results[best_name]['R²']:.4f}")
print(f"  RMSE: {results[best_name]['RMSE']:.4f}")
print(f"  MAE : {results[best_name]['MAE']:.4f}")
print("=" * 50)

joblib.dump(best_model, os.path.join(MODELS_DIR, 'best_model.joblib'))
print(f"\nBest model saved as: models/best_model.joblib")